# ABIDE revision — Stage 2: frozen-design reanalysis

Attach the **same two datasets** used in Stage 1. This notebook is self-contained;
there is no need to attach the Stage 1 output as an additional dataset.
Run all cells on CPU and preserve the full output folder.

## Locked revision-stage design

- Primary demographic criteria: males in the existing dataset, age **18–30 inclusive**,
  observed **FIQ >80**, with no IQ imputation for eligibility: **140 people, 69 ASD/71 controls**.
  Handedness is described as recorded; participants are not assumed all right-handed.
- Primary fixed node set: **106 CC200 ROIs** with nonconstant time series and ≥50%
  finite in-mask fALFF support in **every** eligible participant, in both pipelines.
  All 140 are retained for feature generation. This is a **coverage-limited common-node
  network**, not an unchanged full-200 whole-brain analysis. Nodal medians are recalculated.
- Sensitivities: 181-ROI broader mask (104 people), and full 200 ROIs with ≥50%
  support/nonconstant time series throughout (69 people). Identical masks and participants
  are used across GSR/no-GSR within each design. The exact IDs/ROIs are embedded below.
- Anatomical network assignment additionally requires ≥50% of parcel voxels within
  liberal Yeo cortical support; tiny intersections do not label an entire parcel.
  The primary mask leaves only one limbic ROI, so within-limbic features are unavailable.
- No energy trimming in primary inference. The historical-style pooled-energy rule
  is a sensitivity only, followed by the same small-site rule.
- Primary model: raw H adjusted for **site, age, FIQ, mean FD**, with HC3 95% CI.
  Sites with fewer than three design participants are omitted from adjusted models:
  **136 people, 67 ASD/69 controls**, for the primary model. Every omission is recorded.
- Two primary H tests (GSR/no-GSR) use Holm correction. Secondary Whole/Intra/Inter
  families use BH across both pipelines within each design/model, excluding primary
  H and the legacy bipolarity descriptor. Sensitivity designs are exploratory.

## Included analyses

Corrected ComBat with explicitly protected group/age/IQ/FD is a **sensitivity**, with
no raw fallback. Other outputs include effect sizes/CIs, approximate sensitivity
power, exploratory equivalence bounds, mean-FC/FD-adjusted efficiency relationships,
group-by-efficiency interactions, per-subject null z-scores, clinical correlations,
site-held-out nested PLS regression, exploratory PLSC, and nested site-held-out
classification. No diagnosis-protected ComBat is used for prediction.
The upstream 106-node QC mask uses the whole eligible cohort, so prediction is
exploratory conditional on that mask, not fully prospective workflow validation.

The constrained null permutes states within network/hemisphere blocks: it preserves
coarse organization but **not full spatial autocorrelation**. Equivalence bounds are
conventional ±0.2 pooled SD and exploratory, not a validated clinical SESOI.
PCP fALFF band/input provenance still needs confirmation before final manuscript claims.
These are openly documented **revision-stage choices**, not an original preregistration.

## Run

1. Enable Internet for installing any missing packages; GPU is unnecessary.
2. Run all cells. Feature generation and local efficiency can take time. Progress
   and feature CSVs are checkpointed; no input dataset is modified.
3. Download **`/kaggle/working/abide_stage2_review.zip`** and place it under local
   `results/`. Preserve the full output folder, including arrays, separately.
4. A completed notebook is not automatically a successful analysis. Review the job
   status and ComBat diagnostics. The code stops inference if frozen feature counts fail.


In [ ]:
import importlib.util, subprocess, sys
requirements = {'nibabel': 'nibabel>=5.2,<6', 'statsmodels': 'statsmodels>=0.14,<1',
                'sklearn': 'scikit-learn>=1.3,<2', 'neuroCombat': 'neuroCombat'}
missing = [package for module, package in requirements.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])


In [ ]:
import json
from pathlib import Path
DERIVATIVES_ROOT = Path('/kaggle/input/datasets/saeedrezaeiafshar/abide-full-derivatives-dataset/abide_revision')
PATCH_ROOT = Path('/kaggle/input/datasets/saeedrezaeiafshar/abide-metadata-patch/abide_metadata_patch_local_v2')
OUTPUT_ROOT = Path('/kaggle/working/abide_revision_stage2')
N_PERMUTATIONS = 1000
RUN_COMBAT_SENSITIVITY = True
RUN_CLINICAL = True
RUN_CLASSIFICATION = True

# Frozen from the outcome-blind Stage 1B QC and author-confirmed demographic criteria.
PLAN = json.loads('{"locked_date": "2026-10-25", "selection_basis": "demographic criteria and signal coverage only; no corrected group-effect tests used", "coverage_threshold": 0.5, "primary_ROI_qualification": "nonconstant ROI time series and >=50% finite in-mask support in every eligible participant, in both pipelines", "cortical_assignment_minimum_fraction": 0.5, "demographic_eligible_SUB_IDs": [50499, 50694, 51020, 51470, 51025, 50232, 50490, 50704, 50514, 51467, 50530, 50236, 51585, 51016, 51468, 50134, 51021, 51459, 50702, 50024, 51026, 50012, 50482, 50689, 50004, 51023, 50711, 51318, 51329, 50237, 50130, 51027, 51322, 50485, 50254, 50121, 50496, 51029, 51017, 51328, 51474, 50320, 50690, 51466, 50693, 50498, 50487, 50233, 50136, 50696, 50246, 50700, 51015, 50695, 50686, 51457, 51464, 50507, 51607, 50488, 51018, 50649, 50480, 50491, 50020, 50253, 51019, 50527, 51465, 50388, 50442, 50382, 51147, 51067, 50440, 50446, 50469, 51150, 50107, 51365, 51113, 51146, 50449, 50444, 50030, 51118, 50032, 50445, 50115, 50060, 50703, 51344, 51151, 51114, 50259, 51334, 51131, 50710, 51132, 50058, 50270, 51152, 51148, 50260, 50261, 50344, 50037, 51153, 51112, 51332, 50706, 50439, 51139, 51346, 51117, 50041, 51364, 50040, 50349, 51156, 50046, 51130, 50117, 50709, 51339, 50441, 51149, 51116, 51347, 50263, 51340, 51335, 51333, 50434, 50707, 50467, 50433, 50463, 51342, 51066], "designs": [{"name": "primary_common_106", "ROI_labels": [2, 3, 4, 6, 7, 8, 11, 13, 14, 17, 18, 19, 20, 22, 24, 25, 29, 33, 34, 35, 37, 38, 40, 45, 46, 48, 50, 55, 56, 58, 59, 60, 61, 64, 66, 67, 69, 70, 73, 76, 79, 81, 83, 84, 88, 89, 93, 94, 95, 98, 105, 106, 107, 114, 115, 116, 117, 119, 121, 122, 123, 127, 128, 129, 132, 135, 137, 138, 141, 144, 146, 147, 148, 149, 151, 152, 153, 154, 156, 157, 161, 162, 164, 165, 166, 168, 171, 172, 173, 174, 177, 178, 179, 180, 182, 184, 185, 186, 187, 189, 190, 191, 194, 196, 199, 200], "SUB_IDs": [50499, 50694, 51020, 51470, 51025, 50232, 50490, 50704, 50514, 51467, 50530, 50236, 51585, 51016, 51468, 50134, 51021, 51459, 50702, 50024, 51026, 50012, 50482, 50689, 50004, 51023, 50711, 51318, 51329, 50237, 50130, 51027, 51322, 50485, 50254, 50121, 50496, 51029, 51017, 51328, 51474, 50320, 50690, 51466, 50693, 50498, 50487, 50233, 50136, 50696, 50246, 50700, 51015, 50695, 50686, 51457, 51464, 50507, 51607, 50488, 51018, 50649, 50480, 50491, 50020, 50253, 51019, 50527, 51465, 50388, 50442, 50382, 51147, 51067, 50440, 50446, 50469, 51150, 50107, 51365, 51113, 51146, 50449, 50444, 50030, 51118, 50032, 50445, 50115, 50060, 50703, 51344, 51151, 51114, 50259, 51334, 51131, 50710, 51132, 50058, 50270, 51152, 51148, 50260, 50261, 50344, 50037, 51153, 51112, 51332, 50706, 50439, 51139, 51346, 51117, 50041, 51364, 50040, 50349, 51156, 50046, 51130, 50117, 50709, 51339, 50441, 51149, 51116, 51347, 50263, 51340, 51335, 51333, 50434, 50707, 50467, 50433, 50463, 51342, 51066], "n_ROIs": 106, "n_subjects": 140}, {"name": "sensitivity_broader_181", "ROI_labels": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 28, 29, 30, 31, 33, 34, 35, 36, 37, 38, 39, 40, 41, 44, 45, 46, 47, 48, 50, 53, 54, 55, 56, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 79, 81, 82, 83, 84, 85, 86, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 121, 122, 123, 124, 125, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 199, 200], "SUB_IDs": [50499, 51025, 50232, 50490, 50704, 50514, 50530, 51016, 50134, 51021, 50702, 50024, 51026, 50012, 50482, 50689, 50004, 51023, 51318, 51329, 50237, 50130, 51027, 50485, 50254, 50121, 50496, 51029, 51328, 50320, 50498, 50233, 50136, 50246, 50700, 51015, 50695, 50507, 51607, 50488, 51018, 50649, 50480, 50491, 50020, 50253, 51019, 50527, 50388, 50442, 50382, 51147, 51067, 50440, 50446, 50469, 51150, 50107, 51365, 51113, 51146, 50449, 50444, 50030, 50445, 50115, 50703, 51151, 51114, 51334, 51131, 50710, 51132, 50058, 50270, 51152, 51148, 50260, 50261, 50344, 50037, 51153, 51112, 51332, 50439, 51139, 51346, 50349, 51156, 50046, 51130, 50117, 50709, 50441, 51149, 51116, 50263, 51335, 51333, 50434, 50467, 50433, 50463, 51342], "n_ROIs": 181, "n_subjects": 104}, {"name": "sensitivity_full200", "ROI_labels": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200], "SUB_IDs": [50499, 50514, 51016, 50134, 51021, 50012, 50482, 50004, 51023, 51318, 51329, 50237, 50130, 51027, 50485, 50254, 50121, 50496, 51328, 50498, 50233, 50136, 51607, 50488, 51018, 50649, 50480, 50491, 50020, 50253, 50527, 50388, 50442, 50382, 50440, 50446, 50469, 51365, 50449, 50444, 50030, 50445, 50115, 51151, 51114, 51334, 51131, 51132, 50058, 50270, 50344, 50037, 51112, 50439, 51139, 51346, 50349, 50046, 50117, 50441, 51116, 50263, 51335, 51333, 50434, 50467, 50433, 50463, 51342], "n_ROIs": 200, "n_subjects": 69}], "primary_model": "raw feature ~ diagnosis + age + FIQ + mean FD + site; HC3 uncertainty", "minimum_site_n_for_adjusted_models": 3, "primary_tests": "Hamiltonian GSR and noGSR; Holm correction across these two tests", "secondary_tests": "BH separately by feature family across both pipelines, excluding primary H and legacy bipolarity", "energy_trimming_primary": false, "handedness": "described as recorded; no claim that all participants are right-handed", "fALFF_provenance_status": "downloaded PCP maps; historical band/input-branch verification remains required"}')
for design in PLAN['designs']:
    print(design['name'], design['n_ROIs'], 'ROIs;', design['n_subjects'], 'participants')


In [ ]:
"""Stage 1: validate ABIDE derivatives, rebuild features, and audit legacy trimming.

No harmonization, participant trimming, hypothesis tests, or classifier fitting.
NiBabel is needed only for image I/O; pure-array operations are testable offline.
"""
import hashlib
import json
import math
import platform
import re
import traceback
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import fft as scipy_fft
from scipy.ndimage import affine_transform
from scipy.sparse.csgraph import shortest_path

NETWORKS = {1: 'Visual', 2: 'SomatoMotor', 3: 'DorsalAttn', 4: 'Salience_VentAttn',
            5: 'Limbic', 6: 'Frontoparietal', 7: 'Default'}
PIPELINES = {'GSR': ('filt_global', 'nofilt_global'),
             'noGSR': ('filt_noglobal', 'nofilt_noglobal')}


def json_save(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, allow_nan=False) + '\n')


def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda: f.read(1024 * 1024), b''):
            h.update(b)
    return h.hexdigest()


def resolve_nifti(path):
    """Locate compressed, plain, or Kaggle-extracted NIfTI files by exact name."""
    path = Path(path)
    if path.name.endswith('.nii.gz'):
        plain = path.with_suffix('')
        compressed = path
    elif path.name.endswith('.nii'):
        plain = path
        compressed = path.with_name(path.name + '.gz')
    else:
        raise ValueError(f'Expected a .nii or .nii.gz path: {path}')
    candidates = [path, compressed, plain, plain / plain.name, compressed / plain.name]
    for candidate in dict.fromkeys(candidates):
        if candidate.is_file():
            return candidate
    raise FileNotFoundError('NIfTI not found; checked: ' + ', '.join(str(p) for p in dict.fromkeys(candidates)))


def spatial_image(path):
    import nibabel as nib
    img = nib.load(str(path))
    if int(img.header['sform_code']) == 0 and int(img.header['qform_code']) == 0:
        raise ValueError(f'No explicit spatial transform: {path}')
    values = np.asanyarray(img.dataobj)
    if values.ndim > 3 and all(s == 1 for s in values.shape[3:]):
        values = values.reshape(values.shape[:3])
    if values.ndim != 3 or not np.isfinite(img.affine).all() or abs(np.linalg.det(img.affine[:3, :3])) < 1e-12:
        raise ValueError(f'Invalid spatial image: {path}')
    return values, img.affine.copy()


def resample_labels(source, source_affine, target_shape, target_affine):
    """Nearest-neighbor sampling in world coordinates; this is not registration."""
    transform = np.linalg.solve(source_affine, target_affine)
    return affine_transform(np.asarray(source, dtype=np.int32), transform[:3, :3],
                            offset=transform[:3, 3], output_shape=tuple(target_shape),
                            order=0, mode='constant', cval=0, prefilter=False)


def integer_labels(values):
    if not np.isfinite(values).all() or not np.array_equal(values, np.rint(values)):
        raise ValueError('Atlas must have finite integer labels')
    labels = np.unique(values).astype(int)
    return labels[labels > 0]


def anatomical_mapping(cc, yeo_on_cc, affine):
    labels = integer_labels(cc)
    if not set(np.unique(yeo_on_cc)).issubset(set(range(8))):
        raise ValueError('Expected original Yeo labels 0–7')
    network_sizes = {k: int(np.sum(yeo_on_cc == k)) for k in NETWORKS}
    rows = []
    for label in labels:
        mask = cc == label
        size = int(mask.sum())
        overlap = np.array([np.sum(mask & (yeo_on_cc == k)) for k in NETWORKS], dtype=int)
        dice = np.array([2 * n / (size + network_sizes[k]) if size + network_sizes[k] else 0
                         for k, n in zip(NETWORKS, overlap)])
        winners = np.flatnonzero(np.isclose(dice, dice.max(), rtol=1e-10, atol=1e-12))
        assigned = int(winners[0] + 1) if dice.max() > 0 and len(winners) == 1 else 0
        centroid = np.argwhere(mask).mean(axis=0)
        xyz = affine[:3, :3] @ centroid + affine[:3, 3]
        row = {'ROI_label': int(label), 'ROI_voxels': size, 'network_id': assigned,
               'network_name': NETWORKS.get(assigned, 'Unassigned'),
               'max_Dice': float(dice.max()), 'Yeo_covered_fraction': float(overlap.sum() / size),
               'assignment_tied': bool(dice.max() > 0 and len(winners) > 1),
               'low_cortical_coverage_flag': bool(overlap.sum() / size < .5),
               'centroid_x_mm': float(xyz[0]), 'centroid_y_mm': float(xyz[1]), 'centroid_z_mm': float(xyz[2])}
        for k in NETWORKS:
            row[f'Dice_{k}'] = float(dice[k - 1])
            row[f'overlap_voxels_{k}'] = int(overlap[k - 1])
        rows.append(row)
    return pd.DataFrame(rows)


def read_timeseries(path, atlas_labels, allow_constant=False):
    with Path(path).open() as f:
        header = f.readline().strip()
    if not header.startswith('#'):
        raise ValueError('Missing ROI label header; ordinal column order will not be assumed')
    tokens = header.replace('#', ' ').split()
    if not tokens or any(not re.fullmatch(r'\d+', t) for t in tokens):
        raise ValueError(f'Unrecognized ROI label header: {header[:120]}')
    labels = np.array([int(t) for t in tokens])
    ts = np.loadtxt(path, comments='#', ndmin=2)
    if ts.shape[1] != len(labels) or len(set(labels)) != len(labels):
        raise ValueError('Duplicate ROI labels or header/data column mismatch')
    if set(labels) != set(atlas_labels):
        missing, extra = set(atlas_labels) - set(labels), set(labels) - set(atlas_labels)
        raise ValueError(f'Time-series/atlas labels differ: missing={sorted(missing)}, extra={sorted(extra)}')
    if ts.shape[0] < 50 or not np.isfinite(ts).all():
        raise ValueError('Time series has <50 volumes or nonfinite values')
    if not allow_constant and np.any(np.ptp(ts, axis=0) == 0):
        raise ValueError('Constant ROI time series; FC would be undefined')
    order = [int(np.flatnonzero(labels == k)[0]) for k in atlas_labels]
    return ts[:, order], header


def roi_falff(map_values, roi_on_map, mask_values, atlas_labels, allow_missing=False):
    if map_values.shape != roi_on_map.shape or map_values.shape != mask_values.shape:
        raise ValueError('Map, ROI, and mask grids differ')
    valid = np.isfinite(map_values) & np.isfinite(mask_values) & (mask_values > .5)
    rows, means = [], []
    for label in atlas_labels:
        voxels = roi_on_map == label
        total = int(voxels.sum())
        included = voxels & valid
        n = int(included.sum())
        mean = float(map_values[included].mean()) if n else np.nan
        means.append(mean)
        rows.append({'ROI_label': int(label), 'voxels_on_map_grid': total,
                     'finite_masked_voxels': n, 'coverage_fraction': n / total if total else 0,
                     'mean_fALFF': mean, 'coverage_below_80pct_flag': bool(total == 0 or n / total < .8)})
    if not allow_missing and not np.isfinite(means).all():
        missing = [int(label) for label, mean in zip(atlas_labels, means) if not np.isfinite(mean)]
        raise ValueError(f'ROIs without finite in-mask fALFF voxels: {missing}')
    return np.array(means), rows


def fc_matrix(ts):
    corr = np.corrcoef(ts.T)
    if not np.isfinite(corr).all():
        raise ValueError('Nonfinite correlations')
    z = np.arctanh(np.clip(corr, -.9999, .9999))
    np.fill_diagonal(z, 0)
    return z


def median_states(values):
    return np.where(values >= np.median(values), 1, -1)


def legacy_ratio(ts):
    frequencies = scipy_fft.fftfreq(len(ts), 2.0)
    low = (np.abs(frequencies) >= .01) & (np.abs(frequencies) <= .08)
    power = np.abs(scipy_fft.fft(ts, axis=0)) ** 2
    return power[low].sum(axis=0) / power.sum(axis=0)


def energy(fc, states):
    i, j = np.triu_indices(len(states), 1)
    return float(-np.sum(fc[i, j] * states[i] * states[j]))


def efficiency(fc, local=False):
    w = np.abs(fc)
    distances = np.full(w.shape, np.inf)
    np.divide(1, w, out=distances, where=w > 0)
    np.fill_diagonal(distances, 0)
    def global_eff(d):
        if len(d) < 2:
            return 0.0
        paths = shortest_path(d, directed=False, method='D')
        inv = np.zeros(paths.shape)
        np.divide(1, paths, out=inv, where=(paths > 0) & np.isfinite(paths))
        return float(inv.sum() / (len(d) * (len(d) - 1)))
    g = global_eff(distances)
    if not local:
        return g, np.nan
    values = []
    for k in range(len(fc)):
        neighbors = np.flatnonzero(w[k] > 0)
        values.append(global_eff(distances[np.ix_(neighbors, neighbors)]))
    return g, float(np.mean(values))


def features(fc, states, assignment, compute_local=True):
    n = len(states)
    i, j = np.triu_indices(n, 1)
    w, products = fc[i, j], states[i] * states[j]
    active = w != 0
    if not active.any():
        raise ValueError('All FC edges are zero')
    result = {'Whole_Energy_global': energy(fc, states)}
    # Retain the original spectral descriptor transparently; its interpretation
    # needs justification before inclusion in an inferential feature family.
    sign = np.sign(fc)
    laplacian = np.diag(np.abs(sign).sum(axis=1)) - sign
    _, vecs = np.linalg.eigh(laplacian)
    partition = np.sign(vecs[:, 1])
    result['Whole_Bipolarity_legacy_spectral'] = float(np.mean(np.sign(w[active]) * partition[i[active]] * partition[j[active]] > 0))
    result['Whole_GlobalEfficiency'], result['Whole_LocalEfficiency'] = efficiency(fc, compute_local)
    definitions = {'agreement': (products == 1) & (w > 0),
                   'disagreement': (products == -1) & (w < 0),
                   'imbalanced_same': (products == 1) & (w < 0),
                   'imbalanced_opp': (products == -1) & (w > 0)}
    for name, mask in definitions.items():
        result['Whole_Prop_' + name] = float(mask.sum() / active.sum())
    block_sum = 0.0
    for a in NETWORKS:
        for b in range(a, 8):
            mask = ((assignment[i] == a) & (assignment[j] == b)) | ((assignment[i] == b) & (assignment[j] == a))
            prefix = f'Intra_{NETWORKS[a]}' if a == b else f'Inter_{NETWORKS[a]}__{NETWORKS[b]}'
            result[prefix + '_AvgWeight'] = float(w[mask].mean()) if mask.any() else np.nan
            result[prefix + '_AvgFalffW'] = float((w[mask] * products[mask]).mean()) if mask.any() else np.nan
            value = float(-np.sum(w[mask] * products[mask]))
            result[prefix + '_EnergyBlock'] = value if mask.any() else np.nan
            block_sum += value
    result.update(QC_MeanFisherZ_FC=float(w.mean()), QC_MeanAbsoluteFisherZ_FC=float(np.abs(w).mean()),
                  QC_ZeroEdge_fraction=float(np.mean(~active)),
                  QC_Energy_involving_unassigned_ROIs=float(result['Whole_Energy_global'] - block_sum))
    return result


def flag_iqr(frame, value_column):
    values = pd.to_numeric(frame[value_column], errors='coerce')
    finite = np.isfinite(values)
    if not finite.any():
        return pd.Series(False, index=frame.index), {'n_finite': 0}
    q1, q3 = values[finite].quantile([.25, .75])
    lo, hi = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
    return finite & ((values < lo) | (values > hi)), {
        'n_finite': int(finite.sum()), 'Q1': float(q1), 'Q3': float(q3),
        'lower_fence': float(lo), 'upper_fence': float(hi), 'pooled_diagnoses': True,
        'quantile_interpolation': 'linear', 'fence_endpoints_included': True}


def derivative(root, file_id, strategy, kind):
    extension = '1D' if kind == 'rois_cc200' else 'nii.gz'
    return Path(root) / 'cpac' / strategy / kind / f'{file_id}_{kind}.{extension}'


def run_stage1(derivatives_root, patch_root, output_root, legacy_retained_ids=None,
               compute_local_efficiency=True, save_fc=True):
    import nibabel as nib
    import scipy
    root, patch, out = map(Path, (derivatives_root, patch_root, output_root))
    out.mkdir(parents=True, exist_ok=True)
    for folder in ('reference', 'features', 'audit', 'participant_arrays'):
        (out / folder).mkdir(exist_ok=True)
    json_save(out / 'run_config.json', {
        'started_utc': datetime.now(timezone.utc).isoformat(), 'derivatives_root': str(root),
        'metadata_patch_root': str(patch), 'primary_Yeo_mask': 'liberal (provisional; mapping audit required)',
        'falff_source': 'downloaded nofilt strategy maps; upstream computation remains to verify',
        'FC_source': 'downloaded filt strategy ROI time series, original length',
        'binarization': 'participant median; values >= median are +1',
        'outliers': 'pooled raw energy IQR flags only; no exclusions',
        'compute_local_efficiency': compute_local_efficiency, 'save_FC': save_fc,
        'packages': {'python': platform.python_version(), 'numpy': np.__version__,
                     'pandas': pd.__version__, 'scipy': scipy.__version__, 'nibabel': nib.__version__}})
    participants = pd.read_csv(root / 'metadata/participants_requested.csv')
    if participants.SUB_ID.duplicated().any() or participants.FILE_ID.duplicated().any():
        raise ValueError('Duplicated source participant identifiers')
    tr = pd.read_csv(patch / 'TR_recovery.csv')
    if tr.SUB_ID.duplicated().any() or set(tr.SUB_ID) != set(participants.SUB_ID):
        raise ValueError('TR metadata IDs do not match requested participants')
    participants = participants.merge(tr[['SUB_ID', 'TR_seconds']], on='SUB_ID', how='left', validate='one_to_one')
    if not (pd.to_numeric(participants.TR_seconds, errors='coerce') > 0).all():
        raise ValueError('Every requested participant needs a verified positive TR')
    participants.to_csv(out / 'audit/participants_requested.csv', index=False)
    cc_path = resolve_nifti(root / 'atlases/CC200.nii.gz')
    cc, cc_affine = spatial_image(cc_path)
    labels = integer_labels(cc)
    reference_hashes = {'CC200': {'path': str(cc_path), 'sha256': digest(cc_path)}}
    mappings = {}
    for variant, suffix in [('thin', ''), ('liberal', '_LiberalMask')]:
        name = f'Yeo2011_7Networks_MNI152_FreeSurferConformed1mm{suffix}.normalized.nii.gz'
        path = resolve_nifti(patch / 'atlases/Yeo7_mirror' / name)
        values, affine = spatial_image(path)
        sampled = resample_labels(values, affine, cc.shape, cc_affine)
        nib.save(nib.Nifti1Image(sampled.astype(np.int16), cc_affine), str(out / f'reference/Yeo7_{variant}_on_CC200.nii.gz'))
        mapping = anatomical_mapping(cc, sampled, cc_affine)
        mapping.to_csv(out / f'reference/CC200_Yeo7_mapping_{variant}.csv', index=False)
        mappings[variant] = mapping
        reference_hashes[variant] = {'path': str(path), 'sha256': digest(path)}
    json_save(out / 'reference/source_hashes.json', reference_hashes)
    comparison = mappings['liberal'][['ROI_label', 'network_id']].merge(
        mappings['thin'][['ROI_label', 'network_id']], on='ROI_label', suffixes=('_liberal', '_thin'))
    comparison['assignment_differs'] = comparison.network_id_liberal != comparison.network_id_thin
    comparison.to_csv(out / 'reference/mapping_mask_comparison.csv', index=False)
    assignment = mappings['liberal'].set_index('ROI_label').loc[labels, 'network_id'].to_numpy()
    print(f'Atlas labels: {len(labels)}; unassigned ROIs: {(assignment == 0).sum()}; '
          f'thin/liberal disagreements: {comparison.assignment_differs.sum()}', flush=True)
    rows, audit, coverage, sources, legacy_rows, state_comparisons = [], [], [], [], [], []
    for pipeline, (filtered, unfiltered) in PIPELINES.items():
        for number, participant in enumerate(participants.to_dict('records'), 1):
            sid, fid = int(participant['SUB_ID']), str(participant['FILE_ID'])
            record = {'pipeline': pipeline, 'SUB_ID': sid, 'FILE_ID': fid,
                      'SITE_ID': participant['SITE_ID'], 'DX_GROUP': int(participant['DX_GROUP']),
                      'TR_seconds': float(participant['TR_seconds']), 'status': 'failed', 'error': ''}
            try:
                ts_path = derivative(root, fid, filtered, 'rois_cc200')
                ts, roi_header = read_timeseries(ts_path, labels)
                fc = fc_matrix(ts)
                old_states = median_states(legacy_ratio(ts))
                old_h = energy(fc, old_states)
                legacy_rows.append({'pipeline': pipeline, 'SUB_ID': sid, 'DX_GROUP': int(participant['DX_GROUP']),
                                    'SITE_ID': participant['SITE_ID'], 'legacy_fixed_TR_2_H': old_h})
                record.update(n_volumes=len(ts), n_ROIs=len(labels), roi_header=roi_header,
                              duration_seconds=len(ts) * float(participant['TR_seconds']))
                mask_path = resolve_nifti(derivative(root, fid, unfiltered, 'func_mask'))
                mask, mask_affine = spatial_image(mask_path)
                falff_path = resolve_nifti(derivative(root, fid, unfiltered, 'falff'))
                image, image_affine = spatial_image(falff_path)
                # Masks and CC200 labels use nearest-neighbor interpolation; fALFF values are never resampled.
                roi_grid = resample_labels(cc, cc_affine, image.shape, image_affine)
                mask_grid = resample_labels(mask > .5, mask_affine, image.shape, image_affine)
                falff, cov = roi_falff(image, roi_grid, mask_grid, labels)
                for row in cov:
                    coverage.append({'pipeline': pipeline, 'SUB_ID': sid, **row})
                signs = median_states(falff)
                alt_path = resolve_nifti(derivative(root, fid, filtered, 'falff'))
                alt, alt_affine = spatial_image(alt_path)
                alt_grid = resample_labels(cc, cc_affine, alt.shape, alt_affine)
                alt_mask = resample_labels(mask > .5, mask_affine, alt.shape, alt_affine)
                alt_values, _ = roi_falff(alt, alt_grid, alt_mask, labels)
                state_comparisons.append({'pipeline': pipeline, 'SUB_ID': sid,
                    'filt_nofilt_fALFF_file_hash_identical': digest(falff_path) == digest(alt_path),
                    'filt_nofilt_ROI_max_abs_difference': float(np.max(np.abs(falff - alt_values))),
                    'filt_nofilt_nodal_state_disagreements': int(np.sum(signs != median_states(alt_values))),
                    'downloaded_map_vs_legacy_state_disagreements': int(np.sum(signs != old_states))})
                feat = features(fc, signs, assignment, compute_local_efficiency)
                rows.append({'pipeline': pipeline, 'SUB_ID': sid, 'FILE_ID': fid,
                             'SITE_ID': participant['SITE_ID'], 'DX_GROUP': int(participant['DX_GROUP']), **feat})
                saved = {'ROI_labels': labels, 'fALFF': falff, 'states': signs, 'network_ids': assignment,
                         'legacy_states_fixed_TR_2': old_states}
                if save_fc:
                    saved['FC_FisherZ'] = fc
                np.savez_compressed(out / f'participant_arrays/{pipeline}_{sid}.npz', **saved)
                for kind, path in [('filtered_ROI_time_series', ts_path), ('unfiltered_strategy_fALFF', falff_path),
                                   ('filtered_strategy_fALFF_comparison', alt_path), ('unfiltered_strategy_mask', mask_path)]:
                    sources.append({'pipeline': pipeline, 'SUB_ID': sid, 'kind': kind, 'path': str(path),
                                    'size_bytes': path.stat().st_size, 'sha256': digest(path)})
                record.update(status='ok', n_positive_states=int((signs > 0).sum()),
                              n_median_ties=int(np.sum(falff == np.median(falff))),
                              fALFF_min=float(falff.min()), fALFF_max=float(falff.max()),
                              fALFF_outside_0_1_flag=bool(np.any((falff < 0) | (falff > 1))),
                              n_ROIs_coverage_below_80pct=sum(r['coverage_below_80pct_flag'] for r in cov))
            except Exception as exc:
                record['error'] = f'{type(exc).__name__}: {exc}'
                record['traceback'] = traceback.format_exc()
                print(f'{pipeline} {sid}: {record["error"]}', flush=True)
            audit.append(record)
            if number % 5 == 0 or number == len(participants):
                pd.DataFrame(audit).to_csv(out / 'audit/participant_processing.csv', index=False)
                pd.DataFrame(rows).to_csv(out / 'features/features_all_available_untrimmed.csv', index=False)
                print(f'{pipeline} {number}/{len(participants)}; successful features so far: {len(rows)}', flush=True)
    processing = pd.DataFrame(audit)
    pd.DataFrame(coverage).to_csv(out / 'audit/fALFF_ROI_coverage.csv', index=False)
    pd.DataFrame(sources).to_csv(out / 'audit/input_file_hashes.csv', index=False)
    pd.DataFrame(state_comparisons).to_csv(out / 'audit/fALFF_strategy_comparison.csv', index=False)
    all_features = pd.DataFrame(rows)
    old = pd.DataFrame(legacy_rows)
    fences, flags = [], []
    for pipeline in PIPELINES:
        legacy = old.loc[old.pipeline == pipeline].copy() if len(old) else pd.DataFrame()
        candidate = all_features.loc[all_features.pipeline == pipeline].copy() if len(all_features) else pd.DataFrame()
        if len(candidate):
            flagged, limits = flag_iqr(candidate, 'Whole_Energy_global')
            candidate['pooled_energy_IQR_outlier_flag'] = flagged
            candidate.to_csv(out / f'features/features_{pipeline}_untrimmed.csv', index=False)
            fences.append({'pipeline': pipeline, 'definition': 'downloaded_map_candidate', **limits})
            flags.append(candidate[['pipeline', 'SUB_ID', 'pooled_energy_IQR_outlier_flag']])
        if len(legacy):
            flagged, limits = flag_iqr(legacy, 'legacy_fixed_TR_2_H')
            legacy['legacy_pooled_IQR_outlier_flag'] = flagged
            retained = set(legacy_retained_ids.get(pipeline, [])) if legacy_retained_ids else set()
            legacy['historical_post_IQR_export_retained'] = legacy.SUB_ID.isin(retained) if retained else pd.NA
            legacy['reconstructed_vs_historical_mask_agrees'] = (~flagged == legacy.historical_post_IQR_export_retained) if retained else pd.NA
            legacy.to_csv(out / f'audit/legacy_trimming_reconstruction_{pipeline}.csv', index=False)
            fences.append({'pipeline': pipeline, 'definition': 'legacy_fixed_TR_2', **limits})
    pd.DataFrame(fences).to_csv(out / 'audit/IQR_fences_no_exclusions.csv', index=False)
    if flags:
        pd.concat(flags).to_csv(out / 'audit/outlier_flags_no_exclusions.csv', index=False)
    good_ids = {p: set(processing.loc[(processing.pipeline == p) & (processing.status == 'ok'), 'SUB_ID']) for p in PIPELINES}
    common = set.intersection(*good_ids.values())
    participants.assign(technical_features_available_both_pipelines=participants.SUB_ID.isin(common)).to_csv(
        out / 'audit/common_participant_ledger.csv', index=False)
    summary = {'finished_utc': datetime.now(timezone.utc).isoformat(), 'requested_subjects': len(participants),
        'successful_feature_rows': len(rows), 'expected_feature_rows': 2 * len(participants),
        'technical_failures': int((processing.status != 'ok').sum()),
        'feature_rows_by_pipeline': {p: len(ids) for p, ids in good_ids.items()},
        'common_technical_participants': len(common), 'atlas_ROI_count': len(labels),
        'unassigned_ROIs_liberal': int((assignment == 0).sum()),
        'thin_liberal_mapping_disagreements': int(comparison.assignment_differs.sum()),
        'technical_extraction_complete': len(rows) == 2 * len(participants),
        'scientific_cohort_approved': False, 'upstream_fALFF_validity_established': False,
        'outliers_removed': 0, 'harmonization_performed': False, 'group_tests_performed': False}
    json_save(out / 'stage1_summary.json', summary)
    print(json.dumps(summary, indent=2), flush=True)
    return summary


In [ ]:
"""Frozen-design revision features and inference. Helpers from abide_stage1 precede this in the notebook."""
if 'spatial_image' not in globals():
    from abide_stage1 import *
from scipy import stats
from scipy.optimize import brentq
import importlib.metadata


def adjusted_pvalues(values, method='BH'):
    p = np.asarray(values, float)
    result = np.full(len(p), np.nan)
    valid = np.flatnonzero(np.isfinite(p))
    if not len(valid):
        return result
    order = valid[np.argsort(p[valid])]
    m = len(order)
    if method == 'Holm':
        adjusted = np.maximum.accumulate(p[order] * np.arange(m, 0, -1))
    elif method == 'BH':
        adjusted = np.minimum.accumulate((p[order] * m / np.arange(1, m + 1))[::-1])[::-1]
    else:
        raise ValueError(method)
    result[order] = np.minimum(adjusted, 1)
    return result


def model_design(frame, include_group=True):
    columns = {}
    if include_group:
        columns['diagnosis_ASD'] = frame.DX_GROUP.eq(1).astype(float)
    for name in ('AGE_AT_SCAN', 'FIQ', 'func_mean_fd'):
        values = pd.to_numeric(frame[name], errors='coerce')
        if not np.isfinite(values).all() or values.eq(-9999).any():
            raise ValueError(f'Missing/sentinel model covariate: {name}')
        if values.std(ddof=0) > 0:
            columns[name] = (values - values.mean()) / values.std(ddof=0)
    design = pd.DataFrame(columns, index=frame.index)
    sites = pd.get_dummies(frame.SITE_ID.astype(str), prefix='site', drop_first=True, dtype=float)
    design = pd.concat([pd.Series(1., index=frame.index, name='intercept'), design, sites], axis=1)
    if np.linalg.matrix_rank(design.to_numpy()) != design.shape[1]:
        raise ValueError('Rank-deficient covariate/site design; inspect model manifest')
    if len(design) <= design.shape[1] + 3:
        raise ValueError('Insufficient residual degrees of freedom')
    return design


def hc3_fit(y, design, term='diagnosis_ASD'):
    if np.linalg.matrix_rank(design.to_numpy(dtype=float)) != design.shape[1]:
        raise ValueError('Rank-deficient regression design')
    try:
        import statsmodels.api as sm
    except ImportError:
        # Same OLS/HC3 estimator as the statsmodels matrix check run on Kaggle.
        # This implementation is explicitly identified in the result, not a
        # replacement analysis or a different covariance estimator.
        matrix = design.to_numpy(dtype=float)
        values = np.asarray(y, float)
        inv = np.linalg.inv(matrix.T@matrix)
        beta = np.linalg.lstsq(matrix, values, rcond=None)[0]
        residual = values-matrix@beta
        leverage = np.sum((matrix@inv)*matrix, axis=1)
        if np.max(leverage) >= 1-1e-9:
            raise ValueError('Unit leverage makes HC3 uncertainty undefined')
        weighted = matrix*(residual/(1-leverage))[:, None]
        cov = inv@weighted.T@weighted@inv
        index = design.columns.get_loc(term)
        se = float(np.sqrt(cov[index, index]))
        df = len(values)-matrix.shape[1]
        critical = stats.t.ppf(.975, df)
        return {'beta': float(beta[index]), 'SE_HC3': se,
                'CI95_low': float(beta[index]-critical*se), 'CI95_high': float(beta[index]+critical*se),
                'p': float(2*stats.t.sf(abs(beta[index]/se), df)), 'df_residual': float(df),
                'implementation_backend': 'numpy_OLS_HC3_matrix'}, None
    fit = sm.OLS(np.asarray(y, float), design.astype(float)).fit(cov_type='HC3', use_t=True)
    if np.max(fit.get_influence().hat_matrix_diag) >= 1 - 1e-9:
        raise ValueError('Unit leverage makes HC3 uncertainty undefined')
    ci = fit.conf_int(alpha=.05).loc[term]
    return {'beta': float(fit.params[term]), 'SE_HC3': float(fit.bse[term]),
            'CI95_low': float(ci.iloc[0]), 'CI95_high': float(ci.iloc[1]),
            'p': float(fit.pvalues[term]), 'df_residual': float(fit.df_resid),
            'implementation_backend': 'statsmodels_OLS_HC3'}, fit


def hedge_g(a, b):
    a, b = np.asarray(a), np.asarray(b)
    pooled = np.sqrt(((len(a)-1)*a.var(ddof=1)+(len(b)-1)*b.var(ddof=1))/(len(a)+len(b)-2))
    if pooled <= 0:
        return np.nan
    return float((1-3/(4*(len(a)+len(b))-9)) * (a.mean()-b.mean()) / pooled)


def bootstrap_g(a, b, n_boot=2000, seed=101):
    rng = np.random.default_rng(seed)
    values = [hedge_g(rng.choice(a, len(a), replace=True), rng.choice(b, len(b), replace=True)) for _ in range(n_boot)]
    return np.nanpercentile(values, [2.5, 97.5])


def minimum_detectable_d(n1, n0, alpha=.05, power=.8):
    if min(n1, n0) < 2 or not 0 < alpha < power < 1:
        raise ValueError('Invalid sample sizes, alpha, or target power')
    df = n1+n0-2
    critical = stats.t.ppf(1-alpha/2, df)
    def detection(d):
        nc = d*np.sqrt(n1*n0/(n1+n0))
        # Some SciPy/Boost versions return NaN from nct.sf at large noncentrality.
        # CDF complements plus an adaptive small bracket avoid that unstable tail.
        value = 1-stats.nct.cdf(critical, df, nc)+stats.nct.cdf(-critical, df, nc)
        if not np.isfinite(value):
            raise ValueError(f'Nonfinite noncentral-t power at effect size {d}')
        return float(np.clip(value, 0, 1))
    upper = .1
    while detection(upper) < power:
        upper *= 2
        if upper > 10:
            raise ValueError('Could not bracket sensitivity-power solution')
    return float(brentq(lambda d: detection(d)-power, 0, upper))


def primary_effect_diagnostics(frame, fit_result, fit, seed=101):
    a = frame.loc[frame.DX_GROUP.eq(1), 'Whole_Energy_global'].to_numpy()
    b = frame.loc[frame.DX_GROUP.eq(2), 'Whole_Energy_global'].to_numpy()
    lo, hi = bootstrap_g(a, b, seed=seed)
    # Exploratory conventional small-effect bounds, not a validated clinical SESOI.
    pooled = np.sqrt(((len(a)-1)*a.var(ddof=1)+(len(b)-1)*b.var(ddof=1))/(len(a)+len(b)-2))
    margin = .2*pooled
    beta, se, df = fit_result['beta'], fit_result['SE_HC3'], fit_result['df_residual']
    tost = max(stats.t.sf((beta+margin)/se, df), stats.t.cdf((beta-margin)/se, df))
    return {'ASD_n': len(a), 'Control_n': len(b), 'ASD_mean': float(a.mean()), 'Control_mean': float(b.mean()),
            'ASD_SD': float(a.std(ddof=1)), 'Control_SD': float(b.std(ddof=1)),
            'unadjusted_Hedges_g': hedge_g(a, b), 'g_bootstrap_CI95_low': float(lo), 'g_bootstrap_CI95_high': float(hi),
            'TOST_margin_raw_units': float(margin), 'TOST_exploratory_p': float(tost),
            'TOST_bounds_basis': 'plus/minus 0.2 observed pooled SD; exploratory, not clinical equivalence',
            'two_sample_MDE80_alpha05_approx': minimum_detectable_d(len(a), len(b)),
            'two_sample_MDE80_alpha025_approx': minimum_detectable_d(len(a), len(b), alpha=.025),
            'power_model_caveat': 'two-sample t approximation; not adjusted for sites/covariates'}


def feature_family(name):
    if name == 'Whole_Energy_global':
        return 'primary_H'
    if name.startswith('Whole_'):
        return 'secondary_Whole'
    return name.split('_', 1)[0]


def inferential_features(frame):
    return [c for c in frame if c.startswith(('Whole_', 'Intra_', 'Inter_')) and 'legacy' not in c]


def group_models(all_features, metadata, output, minimum_site_n=3, run_combat=True):
    out = Path(output)
    out.mkdir(parents=True, exist_ok=True)
    results, diagnostics, exclusions, combat_logs = [], [], [], []
    for (name, pipeline), table in all_features.groupby(['design', 'pipeline'], sort=False):
        table = table.merge(metadata, on='SUB_ID', validate='one_to_one')
        counts = table.SITE_ID.value_counts()
        retained = table.SITE_ID.map(counts).ge(minimum_site_n)
        for row in table.loc[~retained, ['SUB_ID', 'SITE_ID', 'DX_GROUP']].to_dict('records'):
            exclusions.append({'design': name, 'pipeline': pipeline, 'reason': 'site_N_below_3_for_adjusted_model', **row})
        data = table.loc[retained].reset_index(drop=True)
        data[['SUB_ID', 'SITE_ID', 'DX_GROUP']].to_csv(out / f'model_participants_{name}_{pipeline}.csv', index=False)
        x = model_design(data)
        x.to_csv(out / f'covariate_design_{name}_{pipeline}.csv', index=False)
        columns = inferential_features(data)
        valid = [c for c in columns if np.isfinite(data[c]).all() and data[c].std() > 1e-12]
        for feature in sorted(set(columns)-set(valid)):
            results.append({'design': name, 'pipeline': pipeline, 'model': 'raw_adjusted', 'Feature': feature,
                            'family': feature_family(feature), 'status': 'unavailable_or_constant_not_tested'})
        versions = {'raw_adjusted': data}
        if run_combat:
            try:
                from neuroCombat import neuroCombat
                covars = pd.DataFrame({'SITE': data.SITE_ID.astype(str), 'Group': data.DX_GROUP.eq(1).astype(int),
                                      'Age': data.AGE_AT_SCAN, 'IQ': data.FIQ, 'FD': data.func_mean_fd})
                adjusted = neuroCombat(dat=data[valid].to_numpy().T, covars=covars, batch_col='SITE',
                                       categorical_cols=['Group'], continuous_cols=['Age', 'IQ', 'FD'])['data'].T
                if adjusted.shape != data[valid].shape or not np.isfinite(adjusted).all():
                    raise ValueError('Invalid ComBat result; no raw fallback allowed')
                modified = data.copy()
                modified[valid] = adjusted
                versions['ComBat_sensitivity_adjusted'] = modified
                modified.to_csv(out / f'ComBat_sensitivity_features_{name}_{pipeline}.csv', index=False)
                combat_logs.append({'design': name, 'pipeline': pipeline, 'status': 'ok',
                                    'max_abs_change': float(np.max(np.abs(adjusted-data[valid].to_numpy())))})
            except Exception as exc:
                combat_logs.append({'design': name, 'pipeline': pipeline, 'status': 'failed_no_fallback', 'error': str(exc)})
        for version, data_version in versions.items():
            for feature in valid:
                stats_row, fit = hc3_fit(data_version[feature], x)
                results.append({'design': name, 'pipeline': pipeline, 'model': version, 'Feature': feature,
                                'family': feature_family(feature), 'status': 'ok', 'n': len(data_version), **stats_row})
                if feature == 'Whole_Energy_global':
                    try:
                        diagnostics.append({'design': name, 'pipeline': pipeline, 'model': version,
                                            **primary_effect_diagnostics(data_version, stats_row, fit)})
                    except Exception as exc:
                        diagnostics.append({'design': name, 'pipeline': pipeline, 'model': version,
                                            'diagnostic_error': str(exc)})
            pd.DataFrame(results).to_csv(out / 'group_models_in_progress.csv', index=False)
        # Pooled-energy trimming is sensitivity only, recalculated within this complete design.
        flagged, fences = flag_iqr(table, 'Whole_Energy_global')
        fences['stage'] = 'all eligible design participants before small-site exclusion'
        trim = table.loc[~flagged].copy()
        # Sites can become singletons after trimming; apply the same minimum-size rule again.
        trim = trim.loc[trim.SITE_ID.map(trim.SITE_ID.value_counts()).ge(minimum_site_n)].reset_index(drop=True)
        trim[['SUB_ID', 'SITE_ID', 'DX_GROUP']].to_csv(out / f'trim_sensitivity_participants_{name}_{pipeline}.csv', index=False)
        if len(trim) >= 20 and trim.DX_GROUP.nunique() == 2:
            try:
                stats_row, fit = hc3_fit(trim.Whole_Energy_global, model_design(trim))
                results.append({'design': name, 'pipeline': pipeline, 'model': 'energy_trim_sensitivity',
                                'Feature': 'Whole_Energy_global', 'family': 'primary_H', 'status': 'ok', 'n': len(trim), **stats_row})
            except ValueError as exc:
                exclusions.append({'design': name, 'pipeline': pipeline, 'reason': 'trim_model_unavailable', 'error': str(exc)})
        json_save(out / f'trim_fences_{name}_{pipeline}.json', fences)
    table = pd.DataFrame(results)
    table['p_adjusted'] = np.nan
    table['correction'] = ''
    for key, subset in table.loc[table.status.eq('ok')].groupby(['design', 'model', 'family']):
        method = 'Holm' if key[2] == 'primary_H' else 'BH'
        table.loc[subset.index, 'p_adjusted'] = adjusted_pvalues(subset.p, method)
        table.loc[subset.index, 'correction'] = method + '_across_both_pipelines_within_family'
    table.to_csv(out / 'group_models.csv', index=False)
    pd.DataFrame(diagnostics).to_csv(out / 'primary_effect_sizes_power_equivalence.csv', index=False)
    pd.DataFrame(exclusions).to_csv(out / 'model_exclusions.csv', index=False)
    pd.DataFrame(combat_logs).to_csv(out / 'ComBat_diagnostics.csv', index=False)
    return table


def residual_partial_correlation(y, z, controls):
    controls = np.asarray(controls, float)
    rank = np.linalg.matrix_rank(controls)
    if len(y) <= rank+2:
        raise ValueError('Insufficient partial-correlation degrees of freedom')
    a = np.asarray(y)-controls@np.linalg.lstsq(controls, y, rcond=None)[0]
    b = np.asarray(z)-controls@np.linalg.lstsq(controls, z, rcond=None)[0]
    r = float(np.corrcoef(a, b)[0, 1])
    df = len(y)-rank-1
    p = float(2*stats.t.sf(abs(r)*np.sqrt(df/max(1-r*r, 1e-15)), df))
    return r, p, df


def efficiency_relationships(primary, metadata, output):
    rows, interactions = [], []
    for pipeline, table in primary.groupby('pipeline'):
        data = table.merge(metadata, on='SUB_ID', validate='one_to_one')
        data = data.loc[data.SITE_ID.map(data.SITE_ID.value_counts()).ge(3)].reset_index(drop=True)
        for diagnosis, subset in data.groupby('DX_GROUP'):
            subset = subset.reset_index(drop=True)
            # Drop sites that become singleton within this group before nuisance residualization.
            subset = subset.loc[subset.SITE_ID.map(subset.SITE_ID.value_counts()).ge(2)].reset_index(drop=True)
            try:
                controls = model_design(subset, include_group=False)
                controls['mean_FC'] = subset.QC_MeanFisherZ_FC
                for feature in ('Whole_GlobalEfficiency', 'Whole_LocalEfficiency'):
                    r, p, df = residual_partial_correlation(subset.Whole_Energy_global, subset[feature], controls)
                    rows.append({'pipeline': pipeline, 'DX_GROUP': diagnosis, 'feature': feature,
                                 'n': len(subset), 'partial_r': r, 'p': p, 'df': df,
                                 'controls': 'mean Fisher-z FC, FD, age, IQ, site'})
            except ValueError as exc:
                rows.append({'pipeline': pipeline, 'DX_GROUP': diagnosis, 'error': str(exc)})
        x = model_design(data)
        eff = (data.Whole_GlobalEfficiency-data.Whole_GlobalEfficiency.mean())/data.Whole_GlobalEfficiency.std(ddof=0)
        x['global_efficiency'] = eff
        x['mean_FC'] = data.QC_MeanFisherZ_FC
        x['group_x_efficiency'] = x.diagnosis_ASD*eff
        row, _ = hc3_fit(data.Whole_Energy_global, x, term='group_x_efficiency')
        interactions.append({'pipeline': pipeline, 'n': len(data), **row})
    table = pd.DataFrame(rows)
    if 'p' in table:
        table['p_BH'] = adjusted_pvalues(table.p)
    table.to_csv(Path(output) / 'efficiency_partial_correlations.csv', index=False)
    table = pd.DataFrame(interactions)
    table['p_Holm'] = adjusted_pvalues(table.p, 'Holm')
    table.to_csv(Path(output) / 'group_by_efficiency_interactions.csv', index=False)


def null_energy_statistics(fc, states, blocks=None, permutations=1000, seed=101):
    rng = np.random.default_rng(seed)
    observed = energy(fc, states)
    null = np.empty(permutations)
    for start in range(0, permutations, 100):
        count = min(100, permutations-start)
        vectors = np.tile(states, (count, 1))
        if blocks is None:
            for row in vectors:
                rng.shuffle(row)
        else:
            for block in np.unique(blocks):
                indices = np.flatnonzero(blocks == block)
                for row in vectors:
                    row[indices] = rng.permutation(row[indices])
        null[start:start+count] = -.5*np.sum((vectors@fc)*vectors, axis=1)
    mean, sd = float(null.mean()), float(null.std(ddof=1))
    return {'observed_H': observed, 'null_mean': mean, 'null_SD': sd,
            'z': (observed-mean)/sd if sd > 1e-12 else None,
            'p_lower_energy': (1+int(np.sum(null <= observed)))/(permutations+1) if sd > 1e-12 else None,
            'n_permutations': permutations, 'zero_null_variance': sd <= 1e-12}


def rebuild_stage2(root, patch, out, plan, compute_local=True, n_null=1000):
    import nibabel as nib
    root, patch, out = Path(root), Path(patch), Path(out)
    for folder in ('features', 'audit', 'arrays', 'statistics'):
        (out / folder).mkdir(parents=True, exist_ok=True)
    json_save(out / 'frozen_design.json', plan)
    json_save(out / 'dependency_versions.json', {name: importlib.metadata.version(name)
        for name in ('numpy', 'pandas', 'scipy', 'nibabel', 'statsmodels', 'scikit-learn', 'neuroCombat')})
    metadata = pd.read_csv(root / 'metadata/participants_requested.csv')
    metadata = metadata.loc[metadata.SUB_ID.isin(plan['demographic_eligible_SUB_IDs'])].copy()
    if len(metadata) != len(plan['demographic_eligible_SUB_IDs']) or metadata.SUB_ID.duplicated().any():
        raise ValueError('Frozen demographic manifest does not match inputs')
    if not (metadata.AGE_AT_SCAN.between(18, 30)&metadata.FIQ.gt(80)&metadata.FIQ.ne(-9999)).all():
        raise ValueError('Demographic inclusion criterion mismatch')
    metadata.to_csv(out / 'audit/demographic_manifest.csv', index=False)
    cc, cc_affine = spatial_image(resolve_nifti(root / 'atlases/CC200.nii.gz'))
    labels = integer_labels(cc)
    yeo_path = resolve_nifti(patch / 'atlases/Yeo7_mirror/Yeo2011_7Networks_MNI152_FreeSurferConformed1mm_LiberalMask.normalized.nii.gz')
    yeo, yeo_affine = spatial_image(yeo_path)
    mapping = anatomical_mapping(cc, resample_labels(yeo, yeo_affine, cc.shape, cc_affine), cc_affine)
    mapping.loc[mapping.Yeo_covered_fraction < plan['cortical_assignment_minimum_fraction'], ['network_id', 'network_name']] = [0, 'Unassigned']
    mapping.to_csv(out / 'audit/anatomical_mapping_majority_cortex.csv', index=False)
    indices = {int(k): i for i, k in enumerate(labels)}
    mapping = mapping.set_index('ROI_label')
    output_rows, qc_rows, null_rows, failures, source_rows = [], [], [], [], []
    for pipeline, (filtered, unfiltered) in PIPELINES.items():
        for counter, person in enumerate(metadata.to_dict('records'), 1):
            sid, fid = int(person['SUB_ID']), person['FILE_ID']
            try:
                ts_path = derivative(root, fid, filtered, 'rois_cc200')
                map_path = resolve_nifti(derivative(root, fid, unfiltered, 'falff'))
                mask_path = resolve_nifti(derivative(root, fid, unfiltered, 'func_mask'))
                ts, _ = read_timeseries(ts_path, labels, allow_constant=True)
                raw_map, affine = spatial_image(map_path)
                raw_mask, mask_affine = spatial_image(mask_path)
                for kind, path in [('ROI_time_series', ts_path), ('fALFF', map_path), ('mask', mask_path)]:
                    source_rows.append({'pipeline': pipeline, 'SUB_ID': sid, 'kind': kind,
                                        'path': str(path), 'sha256': digest(path)})
                falff, coverage = roi_falff(raw_map, resample_labels(cc, cc_affine, raw_map.shape, affine),
                    resample_labels(raw_mask > .5, mask_affine, raw_map.shape, affine), labels, allow_missing=True)
                coverage_fraction = np.array([r['coverage_fraction'] for r in coverage])
                for design in plan['designs']:
                    if sid not in design['SUB_IDs']:
                        continue
                    selected = np.array([indices[k] for k in design['ROI_labels']])
                    if not np.isfinite(falff[selected]).all() or np.any(coverage_fraction[selected] < plan['coverage_threshold']):
                        raise ValueError(f"Frozen coverage rule not met: {design['name']}")
                    selected_ts = ts[:, selected]
                    if np.any(np.ptp(selected_ts, axis=0) == 0):
                        raise ValueError(f"Frozen node set has constant time series: {design['name']}")
                    states = median_states(falff[selected])
                    fc = fc_matrix(selected_ts)
                    net = mapping.loc[design['ROI_labels'], 'network_id'].to_numpy(dtype=int)
                    measures = features(fc, states, net, compute_local=compute_local)
                    output_rows.append({'design': design['name'], 'pipeline': pipeline, 'SUB_ID': sid, **measures})
                    qc_rows.append({'design': design['name'], 'pipeline': pipeline, 'SUB_ID': sid,
                                    'n_ROIs': len(selected), 'minimum_coverage': float(coverage_fraction[selected].min()),
                                    'n_positive_states': int(np.sum(states > 0)), 'n_volumes': len(ts)})
                    np.savez_compressed(out / f"arrays/{design['name']}_{pipeline}_{sid}.npz",
                                        ROI_labels=labels[selected], states=states, fALFF=falff[selected], FC_FisherZ=fc)
                    if design['name'].startswith('primary') and n_null:
                        for null_name, blocks in [('node_permutation', None), ('network_hemisphere_blocks',
                                net*3+np.where(mapping.loc[design['ROI_labels'], 'centroid_x_mm'].to_numpy() < 0, 0, 1))]:
                            null_rows.append({'pipeline': pipeline, 'SUB_ID': sid, 'null': null_name,
                                'n_ROIs': len(states), 'n_positive_states': int(np.sum(states > 0)),
                                'delta_H_per_uniform_edge_shift': float(-((states.sum())**2-len(states))/2),
                                **null_energy_statistics(fc, states, blocks, n_null, seed=10000+sid+(pipeline=='GSR'))})
            except Exception as exc:
                failures.append({'pipeline': pipeline, 'SUB_ID': sid, 'error': str(exc), 'traceback': traceback.format_exc()})
                print('FAILED', pipeline, sid, exc, flush=True)
            if counter % 5 == 0 or counter == len(metadata):
                pd.DataFrame(output_rows).to_csv(out / 'features/all_designs_raw_untrimmed.csv', index=False)
                pd.DataFrame(failures, columns=['pipeline', 'SUB_ID', 'error', 'traceback']).to_csv(out / 'audit/failures.csv', index=False)
                print(f'{pipeline} {counter}/{len(metadata)}; feature rows {len(output_rows)}', flush=True)
    pd.DataFrame(qc_rows).to_csv(out / 'audit/rebuilt_measurement_QC.csv', index=False)
    pd.DataFrame(source_rows).to_csv(out / 'audit/input_hashes.csv', index=False)
    pd.DataFrame(null_rows).to_csv(out / 'statistics/participant_null_scores.csv', index=False)
    expected = 2*sum(d['n_subjects'] for d in plan['designs'])
    if failures or len(output_rows) != expected:
        json_save(out / 'stage2_status.json', {'complete': False, 'expected_feature_rows': expected,
            'feature_rows': len(output_rows), 'failures': len(failures), 'statistics_started': False})
        raise RuntimeError('Feature extraction differs from frozen design; inspect saved failures before inference')
    return pd.DataFrame(output_rows), metadata


In [ ]:
"""Clinical and site-held-out predictive complements for the frozen primary design."""
if 'residual_partial_correlation' not in globals():
    from abide_stage2 import *

BEHAVIOR = {'ADI_R_SOCIAL_TOTAL_A': 'ADI_R_Social', 'ADI_R_VERBAL_TOTAL_BV': 'ADI_R_Verbal',
            'ADI_R_ONSET_TOTAL_D': 'ADI_R_Onset', 'ADOS_TOTAL': 'ADOS_Total', 'ADOS_COMM': 'ADOS_Communication'}


def site_group_splits(groups, y, n_splits):
    """Size-balanced, deterministic site-disjoint folds; reject single-class splits."""
    groups, y = np.asarray(groups), np.asarray(y)
    sites, inverse = np.unique(groups, return_inverse=True)
    if len(sites) < n_splits:
        raise ValueError('Not enough sites for requested folds')
    counts = np.bincount(inverse)
    load, assigned = np.zeros(n_splits, dtype=int), np.empty(len(sites), dtype=int)
    for site_index in np.argsort(-counts, kind='stable'):
        fold = int(np.argmin(load))
        assigned[site_index] = fold
        load[fold] += counts[site_index]
    splits = []
    for fold in range(n_splits):
        test = np.flatnonzero(assigned[inverse] == fold)
        train = np.flatnonzero(assigned[inverse] != fold)
        if len(np.unique(y[train])) != 2 or len(np.unique(y[test])) != 2:
            raise ValueError(f'Site-fold {fold} has only one class; no partial-cohort aggregate is valid')
        splits.append((train, test))
    return splits


def clinical_analyses(primary, metadata, output, n_perm=1000):
    from sklearn.impute import SimpleImputer
    from sklearn.feature_selection import VarianceThreshold
    from sklearn.cross_decomposition import PLSRegression
    from sklearn.pipeline import Pipeline
    from sklearn.model_selection import GroupKFold, GridSearchCV
    from sklearn.metrics import r2_score, mean_squared_error
    out = Path(output)
    correlations, partial, predictions, regression, latent, skips = [], [], [], [], [], []
    for pipeline, table in primary.groupby('pipeline'):
        data = table.merge(metadata, on='SUB_ID', validate='one_to_one')
        data = data[data.DX_GROUP.eq(1)].copy()
        feature_cols = [c for c in inferential_features(data) if np.isfinite(data[c]).all() and data[c].std() > 1e-12]
        for source, label in BEHAVIOR.items():
            if source not in data:
                skips.append({'pipeline': pipeline, 'analysis': label, 'reason': 'clinical_column_absent'})
                continue
            values = pd.to_numeric(data[source], errors='coerce')
            data[label] = values.where(values >= 0, np.nan)
            subset = data.loc[data[label].notna()].reset_index(drop=True)
            if len(subset) < 20 or subset[label].std() == 0:
                skips.append({'pipeline': pipeline, 'analysis': label, 'reason': 'n_below_20_or_constant_outcome'})
                continue
            for feature in feature_cols:
                r, p = stats.pearsonr(subset[feature], subset[label])
                rho, sp = stats.spearmanr(subset[feature], subset[label])
                correlations.append({'pipeline': pipeline, 'score': label, 'feature': feature,
                                     'n': len(subset), 'Pearson_r': r, 'Pearson_p': p, 'Spearman_rho': rho, 'Spearman_p': sp})
            try:
                controls = model_design(subset, include_group=False)
                r, p, df = residual_partial_correlation(subset.Whole_Energy_global, subset[label], controls)
                partial.append({'pipeline': pipeline, 'score': label, 'n': len(subset), 'partial_r': r, 'p': p, 'df': df})
            except ValueError as exc:
                skips.append({'pipeline': pipeline, 'analysis': label+'_partial_H', 'reason': str(exc)})
            groups = subset.SITE_ID.to_numpy()
            n_sites = subset.SITE_ID.nunique()
            if n_sites < 4:
                skips.append({'pipeline': pipeline, 'analysis': label+'_PLSR', 'reason': 'fewer_than_4_sites'})
                continue
            x, y = subset[feature_cols].to_numpy(), subset[label].to_numpy()
            outer = GroupKFold(n_splits=min(5, n_sites))
            oof, baseline = np.full(len(y), np.nan), np.full(len(y), np.nan)
            for fold, (train, test) in enumerate(outer.split(x, y, groups)):
                inner = GroupKFold(n_splits=min(3, len(np.unique(groups[train]))))
                splits = list(inner.split(x[train], y[train], groups[train]))
                maximum_components = min(3, x.shape[1], min(len(a)-1 for a, _ in splits))
                if maximum_components < 1 or any(np.std(y[train][a]) == 0 for a, _ in splits):
                    skips.append({'pipeline': pipeline, 'analysis': label+'_PLSR', 'reason': 'inner_training_insufficient_or_constant', 'fold': fold})
                    continue
                pipe = Pipeline([('imputer', SimpleImputer(strategy='median')),
                                 ('variance', VarianceThreshold()), ('pls', PLSRegression(scale=True))])
                search = GridSearchCV(pipe, {'pls__n_components': list(range(1, maximum_components+1))}, cv=splits,
                                      scoring='neg_mean_squared_error', n_jobs=1, error_score='raise')
                search.fit(x[train], y[train], groups=groups[train])
                oof[test] = search.predict(x[test]).ravel()
                baseline[test] = y[train].mean()
                for j in test:
                    predictions.append({'pipeline': pipeline, 'score': label, 'SUB_ID': int(subset.SUB_ID.iloc[j]),
                                        'site': groups[j], 'fold': fold, 'observed': y[j], 'predicted': oof[j],
                                        'fold_training_mean_baseline': baseline[j]})
            if not np.isfinite(oof).all():
                skips.append({'pipeline': pipeline, 'analysis': label+'_PLSR', 'reason': 'incomplete_OOF_predictions_no_aggregate_score'})
                continue
            regression.append({'pipeline': pipeline, 'score': label, 'n': len(y), 'n_sites': n_sites,
                               'OOF_R2': r2_score(y, oof), 'OOF_MSE': mean_squared_error(y, oof),
                               'training_mean_baseline_MSE': mean_squared_error(y, baseline),
                               'validation': 'nested GroupKFold by site; all fitted steps within folds'})
        # Descriptive PLSC with explicit site-constrained permutations.
        for name, scores in [('ADOS_block', ['ADOS_Total', 'ADOS_Communication']), ('all_five_scores', list(BEHAVIOR.values()))]:
            if any(c not in data for c in scores):
                continue
            block = data.dropna(subset=scores).reset_index(drop=True)
            if len(block) < 20:
                skips.append({'pipeline': pipeline, 'analysis': name+'_PLSC', 'reason': 'complete_case_n_below_20'})
                continue
            x = block[feature_cols].to_numpy()
            y = block[scores].to_numpy()
            try:
                controls = model_design(block, include_group=False).to_numpy()
            except ValueError as exc:
                skips.append({'pipeline': pipeline, 'analysis': name+'_PLSC', 'reason': str(exc)})
                continue
            x = x-controls@np.linalg.lstsq(controls, x, rcond=None)[0]
            y = y-controls@np.linalg.lstsq(controls, y, rcond=None)[0]
            x = (x-x.mean(0))/(x.std(0, ddof=1)+1e-12)
            y = (y-y.mean(0))/(y.std(0, ddof=1)+1e-12)
            singular = np.linalg.svd(y.T@x/(len(x)-1), compute_uv=False)
            rng = np.random.default_rng(101)
            ge = np.zeros(len(singular))
            movable = 0
            sites = block.SITE_ID.to_numpy()
            for s in np.unique(sites):
                movable += int(np.sum(sites == s)) if np.sum(sites == s) > 1 else 0
            if movable < 10:
                skips.append({'pipeline': pipeline, 'analysis': name+'_PLSC', 'reason': 'too_few_permutable_within_site_rows'})
                continue
            for _ in range(n_perm):
                indices = np.arange(len(y))
                for site in np.unique(sites):
                    selected = np.flatnonzero(sites == site)
                    indices[selected] = rng.permutation(selected)
                perm = np.linalg.svd(y[indices].T@x/(len(x)-1), compute_uv=False)
                ge += perm >= singular
            for k, value in enumerate(singular):
                latent.append({'pipeline': pipeline, 'block': name, 'n': len(block), 'LV': k+1,
                               'singular_value': value, 'squared_covariance_fraction': value**2/np.sum(singular**2),
                               'permutation_p': (ge[k]+1)/(n_perm+1), 'permutations': n_perm,
                               'method': 'nuisance-residualized PLSC; within-site residual permutations; exploratory exchangeability assumptions'})
    corr = pd.DataFrame(correlations)
    if len(corr):
        corr['Pearson_p_BH'] = adjusted_pvalues(corr.Pearson_p)
        corr['Spearman_p_BH'] = adjusted_pvalues(corr.Spearman_p)
    corr.to_csv(out / 'clinical_all_feature_correlations.csv', index=False)
    part = pd.DataFrame(partial)
    if len(part):
        part['p_BH'] = adjusted_pvalues(part.p)
    part.to_csv(out / 'clinical_H_partial_correlations.csv', index=False)
    pd.DataFrame(regression).to_csv(out / 'clinical_PLSR_site_heldout.csv', index=False)
    pd.DataFrame(predictions).to_csv(out / 'clinical_PLSR_OOF_predictions.csv', index=False)
    table = pd.DataFrame(latent)
    if len(table):
        table['p_BH_exploratory'] = adjusted_pvalues(table.permutation_p)
    table.to_csv(out / 'clinical_PLSC.csv', index=False)
    pd.DataFrame(skips).to_csv(out / 'clinical_unavailable.csv', index=False)


def classifier_analyses(primary, metadata, output):
    from sklearn.model_selection import GridSearchCV
    from sklearn.pipeline import Pipeline
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import StandardScaler
    from sklearn.feature_selection import VarianceThreshold, SelectPercentile, f_classif
    from sklearn.linear_model import LogisticRegression
    from sklearn.svm import SVC
    from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score
    out = Path(output)
    predictions, folds, unavailable, expected_counts = [], [], [], {}
    for pipeline, table in primary.groupby('pipeline'):
        data = table.merge(metadata[['SUB_ID', 'DX_GROUP', 'SITE_ID']], on='SUB_ID', validate='one_to_one').reset_index(drop=True)
        columns = [c for c in inferential_features(data) if np.isfinite(data[c]).all() and data[c].std() > 1e-12]
        x, y, groups = data[columns].to_numpy(), data.DX_GROUP.eq(1).astype(int).to_numpy(), data.SITE_ID.to_numpy()
        outer_splits = site_group_splits(groups, y, 5)
        expected_counts[pipeline] = len(data)
        models = {'LogisticRegression': (LogisticRegression(max_iter=3000, class_weight='balanced'), {'clf__C': [.01, .1, 1, 10]}),
                  'SVM_RBF': (SVC(class_weight='balanced'), {'clf__C': [.1, 1, 10], 'clf__gamma': ['scale']})}
        for model_name, (model, grid) in models.items():
            for fold, (train, test) in enumerate(outer_splits):
                if len(np.unique(y[train])) < 2 or len(np.unique(y[test])) < 2:
                    unavailable.append({'pipeline': pipeline, 'model': model_name, 'fold': fold, 'reason': 'single_class_train_or_test'})
                    continue
                splits = site_group_splits(groups[train], y[train], 3)
                if any(len(np.unique(y[train][a])) < 2 or len(np.unique(y[train][b])) < 2 for a, b in splits):
                    unavailable.append({'pipeline': pipeline, 'model': model_name, 'fold': fold, 'reason': 'single_class_inner_fold'})
                    continue
                pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('variance', VarianceThreshold()),
                                 ('scaler', StandardScaler()), ('select', SelectPercentile(f_classif)), ('clf', model)])
                search = GridSearchCV(pipe, {**grid, 'select__percentile': [25, 50, 100]}, cv=splits,
                                      scoring='balanced_accuracy', n_jobs=1, error_score='raise')
                search.fit(x[train], y[train])
                pred = search.predict(x[test])
                score = search.decision_function(x[test])
                majority = int(np.mean(y[train]) >= .5)
                folds.append({'pipeline': pipeline, 'model': model_name, 'fold': fold, 'n_train': len(train), 'n_test': len(test),
                    'test_sites': '|'.join(sorted(np.unique(groups[test]))),
                    'balanced_accuracy': balanced_accuracy_score(y[test], pred), 'accuracy': accuracy_score(y[test], pred),
                    'F1_ASD_positive': f1_score(y[test], pred, zero_division=0), 'AUC': roc_auc_score(y[test], score),
                    'majority_baseline_accuracy': accuracy_score(y[test], np.repeat(majority, len(test))),
                    'majority_baseline_balanced_accuracy': .5, 'best_inner_params': str(search.best_params_)})
                for position, j in enumerate(test):
                    predictions.append({'pipeline': pipeline, 'model': model_name, 'SUB_ID': int(data.SUB_ID.iloc[j]),
                                        'site': groups[j], 'fold': fold, 'y_true': int(y[j]),
                                        'y_pred': int(pred[position]), 'decision_score': float(score[position])})
    pd.DataFrame(folds).to_csv(out / 'classification_nested_site_folds.csv', index=False)
    pd.DataFrame(predictions).to_csv(out / 'classification_OOF_predictions.csv', index=False)
    pd.DataFrame(unavailable).to_csv(out / 'classification_unavailable_folds.csv', index=False)
    table = pd.DataFrame(folds)
    pred = pd.DataFrame(predictions)
    complete_keys = set()
    if len(pred):
        for key, block in pred.groupby(['pipeline', 'model']):
            if len(block) == expected_counts[key[0]] and block.SUB_ID.nunique() == expected_counts[key[0]]:
                complete_keys.add(key)
    if len(table):
        complete_table = table.loc[[tuple(k) in complete_keys for k in table[['pipeline', 'model']].to_numpy()]]
        complete_table.groupby(['pipeline', 'model'])[['balanced_accuracy', 'accuracy', 'F1_ASD_positive', 'AUC']].agg(
            ['mean', 'std', 'count']).to_csv(out / 'classification_fold_summary.csv')
    # Conditional site bootstrap of fixed OOF predictions; not a full training-process CI.
    intervals = []
    if len(pred):
        for (pipeline, model_name), block in pred.groupby(['pipeline', 'model']):
            if (pipeline, model_name) not in complete_keys:
                intervals.append({'pipeline': pipeline, 'model': model_name, 'n_OOF_predictions': len(block),
                                  'status': 'incomplete_OOF_no_aggregate_score'})
                continue
            sites = block.site.unique()
            rng = np.random.default_rng(303)
            scores = []
            for _ in range(1000):
                sample = pd.concat([block[block.site.eq(site)] for site in rng.choice(sites, len(sites), replace=True)])
                if sample.y_true.nunique() == 2:
                    scores.append(balanced_accuracy_score(sample.y_true, sample.y_pred))
            low, high = np.percentile(scores, [2.5, 97.5]) if scores else (np.nan, np.nan)
            intervals.append({'pipeline': pipeline, 'model': model_name, 'n_OOF_predictions': len(block),
                              'balanced_accuracy': balanced_accuracy_score(block.y_true, block.y_pred),
                              'conditional_site_bootstrap_CI95_low': low, 'conditional_site_bootstrap_CI95_high': high,
                              'valid_bootstrap_replicates': len(scores),
                              'caveat': 'fixed OOF predictions; does not include all model-training uncertainty'})
    pd.DataFrame(intervals).to_csv(out / 'classification_conditional_intervals.csv', index=False)
    json_save(out / 'classification_scope.json', {
        'training_steps': 'imputation, scaling, feature selection, and tuning nested within site-held-out folds',
        'upstream_QC_mask': '106-node mask selected using QC from the whole eligible cohort before CV',
        'interpretation': 'exploratory prediction conditional on the fixed cohort-wide QC mask; not a fully prospective validation of the entire acquisition/QC workflow',
        'fold_construction': 'deterministic size-balanced site-disjoint K-fold: outer 5, inner 3; both classes required in every split',
        'expected_per_pipeline': expected_counts,
        'complete_pipeline_model_pairs': len(complete_keys),
        'ComBat_used_for_prediction': False})


In [ ]:
"""Orchestrate Stage 2 and export one compact review bundle."""
if 'classifier_analyses' not in globals():
    from abide_stage2_predictive import *


def validate_stats_backend():
    """Compare statsmodels HC3 covariance against its matrix formula before long runs."""
    rng = np.random.default_rng(404)
    x = pd.DataFrame({'intercept': np.ones(80), 'diagnosis_ASD': np.tile([0., 1.], 40),
                      'nuisance': rng.normal(size=80)})
    y = x.to_numpy()@np.array([1., 2., .4])+rng.normal(size=80)
    result, fit = hc3_fit(y, x)
    matrix = x.to_numpy()
    inv = np.linalg.inv(matrix.T@matrix)
    coef = inv@matrix.T@y
    residual = y-matrix@coef
    leverage = np.sum((matrix@inv)*matrix, axis=1)
    weighted = matrix*(residual/(1-leverage))[:, None]
    cov = inv@weighted.T@weighted@inv
    np.testing.assert_allclose(result['beta'], coef[1], rtol=1e-10)
    np.testing.assert_allclose(result['SE_HC3'], np.sqrt(cov[1, 1]), rtol=1e-10)
    return {'statsmodels_HC3_matrix_check': 'passed'}


def run_stage2(derivatives_root, patch_root, output_root, plan, n_perm=1000,
               run_combat=True, run_clinical=True, run_classification=True):
    import zipfile
    out = Path(output_root)
    out.mkdir(parents=True, exist_ok=True)
    json_save(out / 'run_settings.json', {'n_permutations': n_perm, 'run_ComBat_sensitivity': run_combat,
        'run_clinical': run_clinical, 'run_classification': run_classification, 'effect_size_bootstraps': 2000,
        'model_primary': 'raw site/age/IQ/FD-adjusted, HC3', 'prediction_harmonization': 'none'})
    json_save(out / 'backend_validation.json', validate_stats_backend())
    if run_classification:
        metadata_preview = pd.read_csv(Path(derivatives_root) / 'metadata/participants_requested.csv')
        metadata_preview = metadata_preview[metadata_preview.SUB_ID.isin(plan['designs'][0]['SUB_IDs'])]
        labels_preview = metadata_preview.DX_GROUP.eq(1).astype(int).to_numpy()
        sites_preview = metadata_preview.SITE_ID.to_numpy()
        fold_preview = []
        for fold, (train, test) in enumerate(site_group_splits(sites_preview, labels_preview, 5)):
            inner = site_group_splits(sites_preview[train], labels_preview[train], 3)
            fold_preview.append({'fold': fold, 'n_train': len(train), 'n_test': len(test),
                                 'ASD_test': int(labels_preview[test].sum()), 'inner_valid_folds': len(inner)})
        json_save(out / 'classification_fold_preflight.json', fold_preview)
    rows, metadata = rebuild_stage2(derivatives_root, patch_root, out, plan, n_null=n_perm)
    primary_name = plan['designs'][0]['name']
    primary = rows[rows.design.eq(primary_name)].copy()
    jobs = [('group_models', lambda: group_models(rows, metadata, out / 'statistics', run_combat=run_combat)),
            ('efficiency_relationships', lambda: efficiency_relationships(primary, metadata, out / 'statistics'))]
    if run_clinical:
        jobs.append(('clinical_analyses', lambda: clinical_analyses(primary, metadata, out / 'statistics', n_perm=n_perm)))
    if run_classification:
        jobs.append(('classification', lambda: classifier_analyses(primary, metadata, out / 'statistics')))
    status = []
    for name, job in jobs:
        print('START', name, flush=True)
        try:
            job()
            status.append({'analysis': name, 'status': 'completed'})
        except Exception as exc:
            status.append({'analysis': name, 'status': 'failed_no_fallback', 'error': str(exc), 'traceback': traceback.format_exc()})
            print('FAILED', name, exc, flush=True)
        json_save(out / 'analysis_job_status.json', status)
    # Descriptive plots only; all unavailable outcomes stay explicit in CSVs.
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    data = primary.merge(metadata[['SUB_ID', 'DX_GROUP']], on='SUB_ID', validate='many_to_one')
    for ax, (pipeline, block) in zip(axes, data.groupby('pipeline')):
        samples = [block.loc[block.DX_GROUP.eq(g), 'Whole_Energy_global'] for g in (1, 2)]
        ax.boxplot(samples)
        ax.set_xticks([1, 2])
        ax.set_xticklabels(['ASD', 'Control'])
        ax.set_title(f'{pipeline}: common {plan["designs"][0]["n_ROIs"]}-ROI H (raw)')
        ax.set_ylabel('Hamiltonian')
    fig.tight_layout()
    fig.savefig(out / 'primary_raw_energy_distributions.png', dpi=150)
    plt.close(fig)
    summary = {'finished_utc': datetime.now(timezone.utc).isoformat(),
               'primary_demographic_n': len(metadata), 'rebuilt_feature_rows': len(rows),
               'analysis_jobs': status, 'all_jobs_completed': all(s['status'] == 'completed' for s in status),
               'statistical_results_reviewed': False, 'ready_for_submission': False,
               'remaining_scientific_checks': ['upstream fALFF band/input provenance',
                    'author review of changed anatomical scope and handedness description',
                    'QC/model diagnostics, sensitivity results, and model identifiability',
                    'rewritten manuscript, figures, supporting information, response letter'],
               'null_model_caveat': 'network/hemisphere blocks preserve coarse organization, not full spatial autocorrelation',
               'equivalence_caveat': 'exploratory conventional bounds; no validated clinical SESOI',
               'ComBat_caveat': 'exploratory sensitivity; uncertainty does not integrate harmonization parameter estimation'}
    summary['prediction_scope'] = 'site-held-out models conditional on cohort-wide QC mask; not prospective validation of mask selection'
    combat_file = out / 'statistics/ComBat_diagnostics.csv'
    if run_combat and combat_file.exists():
        combat = pd.read_csv(combat_file)
        summary['ComBat_sensitivity_failures'] = int(combat.status.ne('ok').sum())
    json_save(out / 'stage2_summary.json', summary)
    (out / 'README.md').write_text(
        '# Stage 2 revision outputs\n\nUse frozen_design.json for cohort/node definitions.\n'
        'Primary inference: raw site/age/IQ/FD-adjusted H, HC3 CI, Holm across two pipelines.\n'
        'Secondary families: BH across both pipelines within each design/model/family.\n'
        'ROI-policy, ComBat, and energy-trimming models are sensitivities, not additional primary claims.\n'
        'Clinical/PLS and classification analyses use the primary common-node design.\n'
        'Classification and PLSR hold out sites; no diagnosis-protected ComBat is used for prediction.\n'
        'Missing clinical outcomes are score-wise exclusions, not imaging exclusions.\n'
        'Uncomputed/structurally undefined features are not null results.\n'
        'Inspect analysis_job_status.json and ComBat_diagnostics.csv even if the notebook finishes.\n')
    archive = out.parent / 'abide_stage2_review.zip'
    with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
        for path in out.rglob('*'):
            if path.is_file() and path.suffix in ('.csv', '.json', '.png', '.md'):
                bundle.write(path, path.relative_to(out))
    print('Return:', archive, flush=True)
    print(json.dumps(summary, indent=2), flush=True)
    return summary


In [ ]:
summary = run_stage2(
    DERIVATIVES_ROOT, PATCH_ROOT, OUTPUT_ROOT, PLAN,
    n_perm=N_PERMUTATIONS, run_combat=RUN_COMBAT_SENSITIVITY,
    run_clinical=RUN_CLINICAL, run_classification=RUN_CLASSIFICATION,
)


In [ ]:
print((OUTPUT_ROOT / 'analysis_job_status.json').read_text())
path = OUTPUT_ROOT / 'statistics/group_models.csv'
if path.exists():
    group_results = pd.read_csv(path)
    display(group_results.loc[group_results.Feature.eq('Whole_Energy_global')])
path = OUTPUT_ROOT / 'statistics/ComBat_diagnostics.csv'
if RUN_COMBAT_SENSITIVITY and path.exists():
    display(pd.read_csv(path))
print('Return:', OUTPUT_ROOT.parent / 'abide_stage2_review.zip')


## Return the review bundle

Put **`abide_stage2_review.zip`** under the local repository's `results/` directory.
It contains the frozen design, cohort/ROI QC, feature CSVs, all statistical tables,
predictions, failure logs, and a descriptive figure; large participant arrays stay
in the full output folder.

We will review primary estimates, uncertainty, model assumptions, and sensitivity
agreement before drafting revised Results, Methods, figures, and the response letter.
The revised scope must explicitly describe the 106-node primary network and the
full-200 sensitivity; the old full-brain claims cannot be reused unchanged.
